## 层和块
- 回顾一下多层感知机

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(size=(2, 20))

net(X)

tensor([[ 0.1260,  0.0900, -0.1986, -0.2387,  0.1096,  0.1160,  0.0324,  0.0930,
          0.1509, -0.0054],
        [-0.1038,  0.0506, -0.1452, -0.2045,  0.0365,  0.1416,  0.0060, -0.0321,
          0.0633,  0.0054]], grad_fn=<AddmmBackward0>)

### nn.Sequential定义了一种特殊的Module

### 自定义块

In [3]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))

### 实例化多层感知机的层

In [4]:
net = MLP()
net(X)

tensor([[ 0.0055,  0.0666, -0.0756, -0.0226, -0.0184, -0.3849,  0.1029,  0.1600,
         -0.0836, -0.1416],
        [ 0.0259,  0.1850, -0.1046, -0.0318,  0.0025, -0.2861,  0.0395,  0.0885,
         -0.1502, -0.0639]], grad_fn=<AddmmBackward0>)

### 顺序块

In [5]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for block in args:
            self._modules[block] = block

    def forward(self,X):
        for block in self._modules.values():
            X = block(X)
        return X
net =  MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

net(X)

tensor([[ 0.0977, -0.0991,  0.0413,  0.0951, -0.1253,  0.2887,  0.1325, -0.1309,
          0.0779, -0.1531],
        [-0.0513, -0.2321, -0.0754,  0.0494,  0.0046,  0.2346,  0.1091, -0.1185,
         -0.0433, -0.2705]], grad_fn=<AddmmBackward0>)

### 在正向传报函数中执行代码

In [11]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)
    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()
    
net = FixedHiddenMLP()
net(X)

tensor(0.1473, grad_fn=<SumBackward0>)

In [12]:
### 混合搭配各种组合块的方法
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)
    def forward(self, X):
        return self.linear(self.net(X))
    
chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.2641, grad_fn=<SumBackward0>)

### 参数管理

In [13]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))

net(X)

tensor([[0.0088],
        [0.0871]], grad_fn=<AddmmBackward0>)

#### 参数访问

In [14]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.0320,  0.3465, -0.0276, -0.1026, -0.0108,  0.0438,  0.2115, -0.1357]])), ('bias', tensor([-0.0431]))])


#### 目标参数

In [17]:
print(type(net[2].bias))
print(net[2].bias.data)
print(net[2].bias)
net[2].weight.grad == None

<class 'torch.nn.parameter.Parameter'>
tensor([-0.0431])
Parameter containing:
tensor([-0.0431], requires_grad=True)


True

#### 一次性访问所有参数

In [19]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [20]:
net.state_dict()['2.bias'].data

tensor([-0.0431])

### 从嵌套块收集函数

In [21]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())
def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block {i}', block1())
    return net
rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[0.4979],
        [0.4979]], grad_fn=<AddmmBackward0>)

In [22]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


### 内置初始化

In [23]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 0.0082,  0.0163,  0.0002, -0.0145]), tensor(0.))

In [24]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

#### 对某些块应用不同的初始化方法

In [25]:
def xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)
        nn.init.zeros_(m.bias)

net[0].apply(xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)


tensor([ 0.4928, -0.0734,  0.7051, -0.3162])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


#### 自定义初始化

In [26]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape) for name, param in m.named_parameters()])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *=m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init ('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
Init ('weight', torch.Size([1, 8])) ('bias', torch.Size([1]))


tensor([[ 0.0000, -8.4883,  0.0000,  0.0000],
        [-5.7927,  0.0000, -6.2139,  0.0000]], grad_fn=<SliceBackward0>)

In [27]:
net[0].weight.data[:] += 1
net[0].weight.data[0,0] = 42
net[0].weight.data[0]

tensor([42.0000, -7.4883,  1.0000,  1.0000])

#### 参数绑定

In [31]:
shared = nn.Linear(8, 8)
net1 = nn.Sequential(nn.Linear(4, 8),  nn.ReLU(),  shared, nn.ReLU(), shared, nn.ReLU(), nn.Linear(8, 4))
net1(X)
print(net1[2].weight.data[0] == net1[4].weight.data[0])
net1[2].weight.data[0,0] = 100
print(net1[2].weight.data[0] == net1[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


### 自定义层
- 构造一个没有任何参数的自定义层

In [2]:
import torch 
import torch.nn.functional as F
from torch import nn

class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, X):
        return X - X.mean()
    
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))


tensor([-2., -1.,  0.,  1.,  2.])

#### 将层作为组件合并到构建更复杂的模型中

In [3]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

Y = net(torch.rand((4, 8)))
Y.mean()

tensor(3.2596e-09, grad_fn=<MeanBackward0>)

#### 带参数的图层

In [5]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.zeros(units,))
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)
    
dense = MyLinear(5, 3)
dense.weight

Parameter containing:
tensor([[ 0.2429, -0.9001,  0.1475],
        [ 0.1061, -0.4548,  0.1534],
        [-0.8585, -0.5095,  1.7213],
        [ 0.8407,  0.1305, -0.5721],
        [-1.7625, -0.7669, -0.1911]], requires_grad=True)

#### 使用自定义层直接执行正向传播计算

In [6]:
dense(torch.rand(size=(2, 5)))

tensor([[0.0000, 0.0000, 1.3277],
        [0.0000, 0.0000, 1.1849]])

#### 使用自定义层构建模型

In [7]:
net = nn.Sequential(nn.Linear(64, 8), MyLinear(8, 4), CenteredLayer())
net(torch.rand((2, 64)))

tensor([[-0.3653,  0.7336,  0.3614, -0.7106],
        [-0.6557,  0.9206,  0.4266, -0.7106]], grad_fn=<SubBackward0>)

### 读写文件
- 加载和保存张量

In [10]:
import torch
from torch import nn
from torch.nn import functional as F

x = torch.arange(4)
torch.save(x, 'x-file')

x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

#### 存储一个张量列表，然后把他们读回内存

In [11]:
y = torch.zeros(4)
torch.save([x, y], 'x-files')
x2, y2 = torch.load('x-files')
x2, y2

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

#### 写入或读取从字符串映射到张量的字典

In [12]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

#### 加载和保存模型参数

In [13]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))
    
net = MLP()
X = torch.rand(size=(2, 20))
Y = net(X)

#### 将模型的参数存储为一个叫做“mlp.params”的文件

In [14]:
torch.save(net.state_dict(), 'mlp.params')

#### 实例化原始多层感知机模型的一个备份。直接读取文件中存储的参数

In [15]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [16]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

##  计算设备

In [17]:
import torch
from torch import nn

torch.device('cpu'), torch.cuda.device('cuda'), torch.cuda.device('cuda:1')

(device(type='cpu'),
 <torch.cuda.device at 0x1d40f840760>)

#### 查询可用GPU数量

In [18]:
torch.cuda.device_count()

1

#### 若无GPU

In [21]:
def try_gpu(i=0):
    '''如果存在，则返回gpu(i)，否则返回cpu'''
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')
def try_all_gpus():
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]"""
    devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()  


(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

#### 查询张量所在的设备

In [22]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

#### 存储在GPU上

In [25]:
x = x.to(try_gpu())
x

tensor([1, 2, 3], device='cuda:0')

#### 神经网络和GPU

In [29]:
X = torch.rand(size=(2, 4)).to(try_gpu())
net = nn.Sequential(nn.Linear(4, 8))
net = net.to(try_gpu())

net(X)

tensor([[ 0.6173,  0.4955,  0.7636,  0.3126, -1.0647, -0.2196,  0.0513, -0.0357],
        [ 0.6305,  0.4168,  0.7909,  0.4325, -1.0355, -0.2916,  0.1323, -0.0810]],
       device='cuda:0', grad_fn=<AddmmBackward0>)